# 60 — The author's reviews, and rule-based scripts scored against them

The author's review is the yardstick. Each input is shown as (a) how often the author agreed with what the pipeline proposed, with a
95 % Wilson interval (finite-population corrected, z = 1.96), and (b) for the three inputs that had been proposed by a reading step
(year, TRL, company map), how a **rule-based script with no model** compares with the same review.

How each value was obtained (stated plainly):
- **Domain flag of the evtol.news pages** — fixed text rules run by script (`viz/frames.py`, `directory_frame`); the author's individual rulings sit in `review/domain_overrides.csv`.
- **Year, TRL, company map** — proposed by a reading step over the public page text, then checked by the author. The scripts below are the rule-based alternatives.

Inputs: the CSVs the review pages export, copied to `review/decisions/`. Nothing here changes a figure.

In [1]:
import subprocess, sys, pathlib
PILLAR = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(PILLAR)); sys.path.insert(0, str(PILLAR / 'notebooks' / '_build' / 'yearscript'))
import os; os.chdir(PILLAR)
def run(script, *a):
    r = subprocess.run([sys.executable, script, *a], capture_output=True, text=True, cwd=PILLAR)
    print(r.stdout + (r.stderr if r.returncode else ''))
print(PILLAR)

/home/vasco/Vasco Workspace/Tese_Vasco_Lnx/eVTOL-Visual-Evaluation/labeling_evaluation


## 1. Agreement with the author, input by input

Domain rules (kept / excluded sides), development year, TRL; company map is a census.

In [2]:
run('notebooks/_build/review_confidence.py', '--dir', 'review/decisions')

                                             input  agree  of  pool N  share  95 % low  95 % high lower bound ≥ 90%
                         TRL (evidence-based rows)     50  50     105  1.000     0.961      1.000               yes
                evtol.news year (kept as proposed)     84  90    1229  0.933     0.865      0.968                NO
domain exclusion (sample, rules as at review time)     61  68     223  0.897     0.821      0.943                NO
         domain: pages KEPT in the domain (sample)     67  88     884  0.761     0.668      0.835                NO

TRL: 1 marked unsure (not counted)
year, kind of date chosen by the author: publication 35, existed_by 15, announced 15, dev_start 10, flight 8, not_stated 7
year: 50 of 90 kept years are only 'existed by' / publication dates (56%)
AAM firm mapping INCOMPLETE: 26 of 43 rows answered (4 changed); no verdict until all 43 are answered
domain contradictions: 4 of 10 reviewed were overridden to in-domain (a named list, n

## 2. Domain flag under the current rules

The rules now include the author's rulings: an electric option counts as electric, **hybrid-electric is not** an electric option, only unmanned pages are UAVs, a "no passengers" demonstrator stays in when the page says it is built for people, plus the author's individual rulings. The table crosses the author's answers on the sample of KEPT pages with what the current rules say.

In [3]:
import pandas as pd
from viz import config, frames
cfg = config.load(); d = frames.directory_frame(cfg).set_index('slug_key')
print('designs in the domain:', int(d.in_domain.sum()), '| multicopters:', int((d.in_domain & d.class5.eq('WM')).sum()),
      '| patent-linked kept:', int(d.loc[d.patented, 'in_domain'].sum()), 'of', int(d.patented.sum()))
print(d.groupby('class5').in_domain.sum().to_dict())
k = pd.read_csv('review/decisions/DOMAIN_INCLUDED_REVIEW_DECISIONS.csv', keep_default_na=False)
k['rules_now_say_in'] = k.slug_key.map(d.in_domain)
print(pd.crosstab(k.decision, k.rules_now_say_in))
print('patent-linked pages the rules now exclude (they are in the patent dataset):')
print(sorted(d[d.patented & ~d.in_domain].index))

designs in the domain: 884 | multicopters: 268 | patent-linked kept: 84 of 97
{'ER': 57, 'HB': 112, 'LC': 158, 'VT': 289, 'WM': 268}
rules_now_say_in  False  True 
decision                      
exclude              21      0
in_domain             3     64
patent-linked pages the rules now exclude (they are in the patent dataset):
['aerial-rider-5a09-r', 'alakai-technologies-skai-prototype', 'amsl-aero-vertiia-production-model-concept', 'amsl-aero-vertiia-tilt-prop', 'ascendance-flight-technologies-atea', 'bellapt', 'doroni-aerospace-h1-x-cargo', 'honda-evtol', 'jaunt-air-mobility-mav55', 'joby-aviation-s4-t', 'sharifzadeh-zero-g-ecruzer', 'transcend-air-vy-400r', 'vertiia']


## 3. Development year: rule-based script vs the author

`year_rule.py` keeps Resources entries and event sentences about the aircraft, drops company-history and future statements, and takes the earliest year. It was developed on the pages the author did **not** review and scored once on the author's sample.

In [4]:
run('notebooks/_build/yearscript/evaluate_year_rule.py')

development set (pages the author did not review, readers' agreed year): exact 0.611, within 1 year 0.697, n=922
HOLD-OUT, the author's reviewed years (n=81 with a year): exact 49/81 = 0.605  95% interval (finite pop 1229) (0.49965600551560274, 0.7013102095491606)
                                          within 1 year 61/81 = 0.753  95% interval (0.6528437951824729, 0.8318394293854855)
script gave no year for 3 of 81



## 4. TRL: rule-based script vs the author

`trl_rule.py`: certification 8; piloted flight 7; flight of a sub-scale model 5; any other flight 6; ground / wind-tunnel / simulation 4; otherwise 2. The rules were written before scoring and not tuned on the answers.

In [5]:
import pandas as pd
from trl_rule import rule_trl
from lib import intervals
t = pd.read_csv(cfg.path('trl_per_aircraft'), keep_default_na=False, dtype=str).set_index('aircraft_id')
r = pd.read_csv('review/decisions/TRL_REVIEW_DECISIONS.csv', dtype=str, keep_default_na=False)
r = r[r.action.isin(['confirmed', 'changed'])].copy()
r['author'] = r.trl.astype(int); r['script'] = r.aircraft_id.map(lambda a: rule_trl(t.loc[a, 'evidence_quote'])); r['reading step'] = r.aircraft_id.map(lambda a: int(t.loc[a, 'trl']))
n = len(r); ex = int((r.script == r.author).sum()); w1 = int(((r.script - r.author).abs() <= 1).sum()); rd = int((r['reading step'] == r.author).sum())
print(f'rows answered: {n}')
print(f'reading step agrees with the author: {rd}/{n} = {rd/n:.3f}  95% {intervals.wilson_fpc_interval(rd, n, 105)}')
print(f'script exact:  {ex}/{n} = {ex/n:.3f}  95% {intervals.wilson_fpc_interval(ex, n, 105)}')
print(f'script within one level: {w1}/{n} = {w1/n:.3f}  95% {intervals.wilson_fpc_interval(w1, n, 105)}')
print(pd.crosstab(r.author, r.script, rownames=['author'], colnames=['script']))

rows answered: 50
reading step agrees with the author: 50/50 = 1.000  95% (0.9609542127493491, 1.0)
script exact:  34/50 = 0.680  95% (0.5805284811612418, 0.765415035428524)
script within one level: 42/50 = 0.840  95% (0.7530766590940259, 0.9003722055755314)
script   2  4  5   6  7
author                 
2       13  0  0   5  0
5        2  1  4   0  1
6        0  0  0  14  0
7        0  0  0   7  3


## 5. AAM Reality Index company map: rule-based script vs the author (all 43 rows)

`ari_rule.py` matches normalised names. Rows the author did not export are blank in the dataset and counted as confirmed blank (author's statement).

In [6]:
run('notebooks/_build/yearscript/ari_rule.py')

                 oem           script                           gold(author)           reading step  script_ok  reading_ok
               Ehang                  Guangzhou Ehang Intelligent Technology                             False       False
     Volant Aerotech                                        Wollant Aviation       Wollant Aviation      False        True
    Eve Air Mobility                                           Embraer / Eve          Embraer / Eve      False        True
           Aerofugia                                          Geely Aviation Wofei / Geely Aviation      False       False
           Pipistrel                                               Pipistrel              Pipistrel      False        True
 Pipistrel (Textron)                                               Pipistrel              Pipistrel      False        True
           Kittyhawk                                              Kitty Hawk             Kitty Hawk      False        True
    Dufour Aeros

## 6. One table: accuracy against the author's review

**How the domain "estimation" works.** The author checked a random sample of each side. If k of n sampled pages were right, the share right in the whole pool is estimated as k/n, with a 95 % Wilson interval corrected for the finite pool. Multiplying the share by the pool size gives an estimated **number of pages**: e.g. 67 of 88 kept pages right in a pool of 1 040 gives about 792 right and about 248 wrong, with an interval around each. The two sides are kept separate because their pools were drawn at different times.

Domain rows are the rules **as they stood when the author reviewed** (an honest test). The rules were fixed afterwards; a fresh sample would be needed to state a figure for the current rules. Year, TRL and company map: "LLM" = the reading step that proposed the value; "script" = the rule-based alternative.

In [7]:
import numpy as np
rows = []
def add(inp, who, k, n, N, what):
    lo, hi = intervals.wilson_fpc_interval(k, n, N) if N else (np.nan, np.nan)
    rows.append({'input': inp, 'method': who, 'agree': k, 'of': n, 'share %': round(100*k/n, 1),
                 '95% interval %': (f'{100*lo:.1f}-{100*hi:.1f}' if N else 'census (all rows)'), 'what it measures': what})
dec_ex = pd.read_csv('review/decisions/DOMAIN_REVIEW_DECISIONS.csv', keep_default_na=False); dec_ex = dec_ex[(dec_ex.section == 'B') & dec_ex.decision.isin(['exclude', 'in_domain'])]
dec_in = pd.read_csv('review/decisions/DOMAIN_INCLUDED_REVIEW_DECISIONS.csv', keep_default_na=False)
add('Domain flag: deleting pages as out of domain', 'script rules', int((dec_ex.decision == 'exclude').sum()), len(dec_ex), 223, 'share of excluded pages that really are out')
add('Domain flag: keeping pages in the domain', 'script rules', int((dec_in.decision == 'in_domain').sum()), len(dec_in), 1040, 'share of kept pages that really are in')
ye = pd.read_csv('review/decisions/YEAR_REVIEW_SAMPLE.csv', keep_default_na=False, dtype=str)
add('Development year', 'LLM (reading step)', int((ye.action == 'confirmed').sum()), len(ye), 1229, 'proposed year kept by the author')
yr = pd.read_csv('review/decisions/YEAR_RULE_VS_AUTHOR.csv')
add('Development year', 'script, exact year', int(yr.exact.sum()), len(yr), 1229, 'script year = the author year')
add('Development year', 'script, within 1 year', int(yr.within1.sum()), len(yr), 1229, 'script year within +-1 of the author')
add('TRL', 'LLM (reading step)', 50, 50, 105, 'proposed TRL kept by the author (evidence-based rows)')
add('TRL', 'script, exact level', ex, n, 105, 'script TRL = the author TRL')
add('TRL', 'script, within 1 level', w1, n, 105, 'script TRL within +-1 of the author')
import ari_rule
from viz import thesis
v_ = thesis.context(cfg).v
nm = sorted({str(x) for x in v_.company_canonical if str(x) not in ('Individual Inventor', 'Unknown / Independent')})
m_ = pd.read_csv(cfg.path('ari_firm_map'), keep_default_na=False)
rv_ = pd.read_csv('review/decisions/ARI_FIRM_REVIEW_DECISIONS.csv', keep_default_na=False).set_index('oem')
gold = [rv_.loc[o, 'company_canonical'] if o in rv_.index else c0 for o, c0 in zip(m_.oem, m_.company_canonical)]
add('AAM index company map (all 43 rows)', 'LLM (reading step)', sum(a == b for a, b in zip(m_.company_canonical, gold)), 43, None, 'proposed name = the author name')
add('AAM index company map (all 43 rows)', 'script, name match', sum(ari_rule.match(o, nm) == g for o, g in zip(m_.oem, gold)), 43, None, 'script name = the author name')
T = pd.DataFrame(rows); T.to_csv('review/decisions/ACCURACY_TABLE.csv', index=False)
from IPython.display import display; display(T)
print()
for side, k_, n_, N_ in [('deleted as out of domain', int((dec_ex.decision == 'exclude').sum()), len(dec_ex), 223), ('kept in the domain', int((dec_in.decision == 'in_domain').sum()), len(dec_in), 1040)]:
    lo, hi = intervals.wilson_fpc_interval(k_, n_, N_)
    print(f'{side}: pool {N_} pages -> estimated RIGHT {round(N_*k_/n_)} ({round(N_*lo)}-{round(N_*hi)}), estimated WRONG {round(N_*(1-k_/n_))} ({round(N_*(1-hi))}-{round(N_*(1-lo))})')

,input,method,agree,of,share %,95% interval %,what it measures
0,Domain flag: deleting pages as out of domain,script rules,61,68,89.7,82.1-94.3,share of excluded pages that really are out
1,Domain flag: keeping pages in the domain,script rules,67,88,76.1,66.7-83.6,share of kept pages that really are in
2,Development year,LLM (reading step),84,90,93.3,86.5-96.8,proposed year kept by the author
3,Development year,"script, exact year",49,81,60.5,50.0-70.1,script year = the author year
4,Development year,"script, within 1 year",61,81,75.3,65.3-83.2,script year within +-1 of the author
5,TRL,LLM (reading step),50,50,100.0,96.1-100.0,proposed TRL kept by the author (evidence-base...
6,TRL,"script, exact level",34,50,68.0,58.1-76.5,script TRL = the author TRL
7,TRL,"script, within 1 level",42,50,84.0,75.3-90.0,script TRL within +-1 of the author
8,AAM index company map (all 43 rows),LLM (reading step),38,43,88.4,census (all rows),proposed name = the author name
9,AAM index company map (all 43 rows),"script, name match",35,43,81.4,census (all rows),script name = the author name



deleted as out of domain: pool 223 pages -> estimated RIGHT 200 (183-210), estimated WRONG 23 (13-40)
kept in the domain: pool 1040 pages -> estimated RIGHT 792 (694-869), estimated WRONG 248 (171-346)


## 7. Reading the numbers

- A script is acceptable as a stand-in when its agreement with the author is at or above the author's threshold; otherwise report it as a transparent baseline.
- The author's threshold for the reading-step inputs was a lower bound of 90 %; the question of whether a script may be judged on "within one year / one level" is the author's to decide.